# Exercise 2: Forecasting the SP500 with fundamental indicators
## Neural Network vs LSTM vs Gaussian Process

**Team:** [TEAM MEMBER NAMES]

**Date:** 2026-05-02

Predict the SP500 monthly series 1927-2021 using the team-assigned Goyal-Welch indicators (`dp`, `svar`, `bm`) plus lags of the indicators and lags of the target. Three target specifications are compared head-to-head:

1. **return** : next-month log return of the index
2. **price** : next-month log price of the index
3. **direction** : sign of next-month log return (binary)

For each target the notebook fits
- a multi-layer perceptron with a hyperparameter grid (8 configurations),
- a Gaussian-process regressor with four kernels (three stationary, one custom non-stationary additive),
- an LSTM with a hyperparameter grid (4 configurations).

Splits are chronological throughout: 60% train, 15% validation, 25% test. The validation block is used only to select hyperparameters; after the best configuration is chosen, every model is refit on the train+validation block before being scored on the held-out test set. Feature scaling and L1 feature selection (LassoCV / LogisticRegressionCV with TimeSeriesSplit inner CV) are fit on the training set only. A sensitivity analysis re-runs the full pipeline with each exogenous variable removed in turn.

## Part 1: Setup, dependencies, GPU detection

In [ ]:
# Environment detection: when executed inside Google Colab the notebook
# mounts Google Drive and reads the dataset from the project directory.
# Locally it reads from the current working directory.
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q tensorflow scikit-learn pandas matplotlib

In [ ]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV, LogisticRegressionCV
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    RBF, Matern, RationalQuadratic, ConstantKernel as C,
    WhiteKernel, DotProduct,
)
from sklearn.metrics import mean_squared_error, accuracy_score, matthews_corrcoef

# GPU detection: TensorFlow uses any available accelerator automatically.
gpus = tf.config.list_physical_devices('GPU')
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
print(f'TensorFlow {tf.__version__}, GPUs detected: {gpus}')

# Reproducibility: every model that uses randomness is seeded.
SEED = 7
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# === Project paths and data location ====================================
# On Colab the project directory lives under /content/drive/MyDrive and
# the data file is read from there. Locally the script reads from the
# current working directory.
DRIVE_FOLDER = 'ML4F_HW1'

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_DIR = Path('/content/drive/MyDrive') / DRIVE_FOLDER
    DATA_FILE = PROJECT_DIR / 'goyal-welch2022Monthly.csv'
    OUTPUT_DIR = PROJECT_DIR / 'output'
else:
    DATA_FILE = Path('goyal-welch2022Monthly.csv')
    OUTPUT_DIR = Path('output')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(DATA_FILE)

# === Constants for the assignment =======================================
START_DATE = '1927-01-01'
END_DATE   = '2021-12-31'
ASSIGNED_INDICATORS = ['dp', 'svar', 'bm']   # team-assigned indicators
N_LAGS_TARGET = 3                            # lags of the target
N_LAGS_EXOG = 3                              # lags of the exogenous variables
P_TRAIN = 0.60   # train fraction (chronological)
P_VAL   = 0.15   # validation fraction (used only for hyperparameter selection)
P_TEST  = 0.25   # held-out test fraction

## Part 2: Load Goyal-Welch and build target series

In [ ]:
print('Loading Goyal-Welch monthly data.')
gw = pd.read_csv(DATA_FILE)
# Convert yyyymm (int) to first-of-month date and set as index
gw['Date'] = pd.to_datetime(gw['yyyymm'].astype(str), format='%Y%m')
gw.set_index('Date', inplace=True)
gw.drop('yyyymm', axis=1, inplace=True)
# 'Index' column is stored with thousands separators; strip and cast
gw['Index'] = gw['Index'].str.replace(',', '', regex=False).astype(float)
# Build the assigned indicators as the helper script defines them
gw['dp'] = np.log(gw['D12']) - np.log(gw['Index'])
gw['bm'] = gw['b/m']
# svar already in the raw file

# Two natural target series. The third (direction) is derived inside
# build_design from the restricted panel, so the t+1 alignment never
# uses observations outside the assignment window 1927-2021.
gw['log_price'] = np.log(gw['Index'])
gw['log_ret']   = gw['log_price'].diff()

panel = gw.loc[START_DATE:END_DATE,
               ['Index', 'log_price', 'log_ret'] + ASSIGNED_INDICATORS].copy()
panel = panel.dropna()
print(f'Panel rows after restriction and NA-drop: {len(panel)}.')
panel.head()

## Part 3: Lagged feature matrix and chronological split

In [ ]:
def build_design(panel: pd.DataFrame, target_kind: str):
    """Build (X, y) with lagged features for one of three target kinds.

    target_kind:
      'return'    -> y = log_ret_{t+1};  x = lags 1..L of log_ret + lags 1..L of dp/svar/bm
      'price'     -> y = log_price_{t+1}; x = lags 1..L of log_price + lags 1..L of dp/svar/bm
      'direction' -> y = 1{log_ret_{t+1} > 0}; x = lags 1..L of log_ret + lags 1..L of dp/svar/bm

    All features are observed at time t (the current month); the target is
    the value at time t+1. Direction is built from panel['log_ret'].shift(-1)
    inside this function so the t+1 alignment never reaches outside the
    panel's date window: the last row's direction is NaN (because
    log_ret_{t+1} is unavailable) and is dropped by the dropna below.
    """
    if target_kind == 'return':
        y = panel['log_ret'].shift(-1)
        own_lag_col = 'log_ret'
    elif target_kind == 'price':
        y = panel['log_price'].shift(-1)
        own_lag_col = 'log_price'
    elif target_kind == 'direction':
        next_ret = panel['log_ret'].shift(-1)
        y = (next_ret > 0).astype(float)        # 0/1 as float so NaN can propagate
        y = y.where(next_ret.notna())            # mark out-of-window targets as NaN
        own_lag_col = 'log_ret'
    else:
        raise ValueError(target_kind)

    cols = {}
    for lag in range(1, N_LAGS_TARGET + 1):
        cols[f'{own_lag_col}_lag{lag}'] = panel[own_lag_col].shift(lag - 1)
    for v in ASSIGNED_INDICATORS:
        for lag in range(1, N_LAGS_EXOG + 1):
            cols[f'{v}_lag{lag}'] = panel[v].shift(lag - 1)

    X = pd.DataFrame(cols, index=panel.index)
    df = pd.concat([X, y.rename('TARGET')], axis=1).dropna()
    X = df.drop(columns='TARGET')
    y = df['TARGET']
    if target_kind == 'direction':
        y = y.astype(int)                        # back to 0/1 ints once NaNs are gone
    return X, y


def chronological_split(X: pd.DataFrame, y: pd.Series):
    n = len(X)
    n_train = int(P_TRAIN * n)
    n_val   = int(P_VAL * n)
    return (X.iloc[:n_train], y.iloc[:n_train],
            X.iloc[n_train:n_train + n_val], y.iloc[n_train:n_train + n_val],
            X.iloc[n_train + n_val:], y.iloc[n_train + n_val:])

## Part 4: Feature selection (Lasso / L1 logistic on TRAIN ONLY)

In [ ]:
from sklearn.model_selection import TimeSeriesSplit

def select_features(X_tr: pd.DataFrame, y_tr: pd.Series, target_kind: str):
    """L1-penalised regression / logistic regression with TIME-SERIES inner
    CV on the training set only. Always retain at least three columns by
    falling back to the top three by absolute coefficient if the L1 solution
    is too sparse. TimeSeriesSplit guarantees that every CV fold validates
    on data strictly after the data it was trained on, removing any hidden
    temporal leakage that the default random KFold would introduce."""
    scaler = StandardScaler().fit(X_tr)
    Xs = scaler.transform(X_tr)
    cv_splits = TimeSeriesSplit(n_splits=5)
    if target_kind == 'direction':
        cv = LogisticRegressionCV(Cs=10, cv=cv_splits, penalty='l1', solver='saga',
                                  max_iter=10000, scoring='accuracy',
                                  random_state=SEED).fit(Xs, y_tr)
        coefs = np.abs(cv.coef_).ravel()
    else:
        cv = LassoCV(cv=cv_splits, max_iter=20000, random_state=SEED).fit(Xs, y_tr)
        coefs = np.abs(cv.coef_)
    keep = [c for c, w in zip(X_tr.columns, coefs) if w > 1e-8]
    if len(keep) < 3:
        order = np.argsort(coefs)[::-1]
        keep = [X_tr.columns[i] for i in order[:3]]
    return keep

## Part 5: Metrics, scaling, and the three model fitters

In [ ]:
def nrmse(y_true, y_pred):
    """Lab metric (PyLab5): NRMSE = sqrt(MSE / mean(sum((y - mean)^2))).
    Smaller is better; lower than 1.0 means we beat the sample-mean baseline."""
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()
    denom = np.mean(np.sum((y_true - y_true.mean()) ** 2))
    return float(np.sqrt(mean_squared_error(y_true, y_pred) / denom))


def pcorrect(y_true, y_pred):
    return (1.0 - nrmse(y_true, y_pred)) * 100.0


def standardise(X_tr, X_va, X_te):
    """Fit StandardScaler on TRAIN ONLY then transform all three splits."""
    sc = StandardScaler().fit(X_tr)
    return sc.transform(X_tr), sc.transform(X_va), sc.transform(X_te), sc


def evaluate(y_true, y_pred, classification: bool) -> dict:
    if classification:
        return {
            'accuracy': accuracy_score(y_true, y_pred),
            'MCC':      matthews_corrcoef(y_true, y_pred),
            'n':        int(len(y_true)),
        }
    return {
        'RMSE':     float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'NRMSE':    nrmse(y_true, y_pred),
        'pcorrect': pcorrect(y_true, y_pred),
        'n':        int(len(y_true)),
    }

In [ ]:
def fit_nn(X_tr, y_tr, X_va, y_va, classification: bool):
    """Tune width/depth/activation on validation; refit best on train+val."""
    grid = []
    for hls in [(64,), (128,), (64, 64), (128, 64)]:
        for act in ['relu', 'tanh']:
            grid.append({'hidden_layer_sizes': hls, 'activation': act})
    Cls = MLPClassifier if classification else MLPRegressor
    best, best_score = None, -np.inf if classification else np.inf
    for k, params in enumerate(grid):
        m = Cls(max_iter=2000, alpha=1e-3, solver='adam',
                random_state=SEED, **params)
        m.fit(X_tr, y_tr)
        if classification:
            s = accuracy_score(y_va, m.predict(X_va))
            better = s > best_score
        else:
            s = nrmse(y_va, m.predict(X_va))
            better = s < best_score
        if better:
            best_score = s
            best = (params, m)
        print(f'    NN[{k+1}/{len(grid)}] {params} -> val={s:.4f}', flush=True)
    print(f'    NN best: {best[0]} (val={best_score:.4f})', flush=True)
    Xfull = np.vstack([X_tr, X_va])
    yfull = np.concatenate([np.asarray(y_tr), np.asarray(y_va)])
    final = Cls(max_iter=2000, alpha=1e-3, solver='adam',
                random_state=SEED, **best[0]).fit(Xfull, yfull)
    return final, best[0]

In [ ]:
def fit_gp(X_tr, y_tr, X_va, y_va):
    """Try four kernels on the validation set; refit best on train+val.
    'custom_RBF+Lin' is a non-stationary additive kernel: an RBF block plus
    a DotProduct (linear) block plus white noise. The three stationary
    alternatives are an RBF, a Matern(2.5) and a RationalQuadratic."""
    kernels = {
        'RBF':            C(1.0) * RBF(1.0) + WhiteKernel(noise_level=1e-2),
        'Matern_2.5':     C(1.0) * Matern(length_scale=1.0, nu=2.5)
                          + WhiteKernel(noise_level=1e-2),
        'RatQuad':        C(1.0) * RationalQuadratic(length_scale=1.0, alpha=1.0)
                          + WhiteKernel(noise_level=1e-2),
        'custom_RBF+Lin': (C(1.0) * RBF(1.0)) + (C(1.0) * DotProduct())
                          + WhiteKernel(noise_level=1e-2),
    }
    best, best_score = None, np.inf
    for name, k in kernels.items():
        gp = GaussianProcessRegressor(kernel=k, alpha=1e-6,
                                      normalize_y=True,
                                      n_restarts_optimizer=2,
                                      random_state=SEED)
        gp.fit(X_tr, y_tr)
        s = nrmse(y_va, gp.predict(X_va))
        if s < best_score:
            best_score = s
            best = (name, gp)
        print(f'    GP[{name}] -> val={s:.4f}', flush=True)
    print(f'    GP best: {best[0]} (val={best_score:.4f})', flush=True)
    Xfull = np.vstack([X_tr, X_va])
    yfull = np.concatenate([np.asarray(y_tr), np.asarray(y_va)])
    final = GaussianProcessRegressor(kernel=best[1].kernel_,
                                     alpha=1e-6, normalize_y=True,
                                     n_restarts_optimizer=2,
                                     random_state=SEED).fit(Xfull, yfull)
    return final, {'kernel': best[0]}

In [ ]:
def to_lstm_tensor(X: np.ndarray, y: np.ndarray, timesteps: int):
    """Pack a flat (N, F) feature matrix into LSTM-shaped (N - T + 1, T, F)
    by sliding a window of length `timesteps`."""
    n = X.shape[0]
    if n < timesteps:
        return np.empty((0, timesteps, X.shape[1])), np.empty((0,))
    Xs, ys = [], []
    for i in range(timesteps - 1, n):
        Xs.append(X[i - timesteps + 1:i + 1, :])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)


def fit_lstm(X_tr, y_tr, X_va, y_va, classification: bool):
    """Tune (units, layers, timesteps) on validation; refit best on train+val."""
    best, best_score = None, -np.inf if classification else np.inf
    grid = [
        {'units': 32, 'layers': 1, 'timesteps': 6},
        {'units': 64, 'layers': 1, 'timesteps': 6},
        {'units': 32, 'layers': 2, 'timesteps': 6},
        {'units': 64, 'layers': 2, 'timesteps': 12},
    ]
    for k_idx, params in enumerate(grid):
        T_ = params['timesteps']
        Xtr3, ytr3 = to_lstm_tensor(X_tr, np.asarray(y_tr), T_)
        # Seed the validation tensor with the last (T_-1) train rows so the
        # first valid window in the validation block is properly seeded
        Xva3, yva3 = to_lstm_tensor(np.vstack([X_tr[-(T_ - 1):], X_va]),
                                    np.concatenate([np.asarray(y_tr)[-(T_ - 1):],
                                                    np.asarray(y_va)]), T_)
        Xva3 = Xva3[-len(y_va):]
        yva3 = yva3[-len(y_va):]

        tf.keras.utils.set_random_seed(SEED)
        m = Sequential()
        m.add(Input(shape=(T_, X_tr.shape[1])))
        for i in range(params['layers']):
            return_seq = i < params['layers'] - 1
            m.add(LSTM(params['units'], return_sequences=return_seq))
            m.add(Dropout(0.1))
        if classification:
            m.add(Dense(1, activation='sigmoid'))
            m.compile(loss='binary_crossentropy', optimizer='adam',
                      metrics=['accuracy'])
        else:
            m.add(Dense(1))
            m.compile(loss='mse', optimizer='adam')
        es = EarlyStopping(monitor='val_loss', patience=10,
                           restore_best_weights=True)
        m.fit(Xtr3, ytr3, epochs=200, batch_size=32, verbose=0,
              validation_split=0.1, callbacks=[es])
        if classification:
            preds = (m.predict(Xva3, verbose=0).ravel() > 0.5).astype(int)
            s = accuracy_score(yva3, preds)
            better = s > best_score
        else:
            preds = m.predict(Xva3, verbose=0).ravel()
            s = nrmse(yva3, preds)
            better = s < best_score
        if better:
            best_score = s
            best = params
        print(f'    LSTM[{k_idx+1}/{len(grid)}] {params} -> val={s:.4f}', flush=True)
    print(f'    LSTM best: {best} (val={best_score:.4f})', flush=True)

    T_ = best['timesteps']
    Xfull = np.vstack([X_tr, X_va])
    yfull = np.concatenate([np.asarray(y_tr), np.asarray(y_va)])
    Xf3, yf3 = to_lstm_tensor(Xfull, yfull, T_)

    tf.keras.utils.set_random_seed(SEED)
    final = Sequential()
    final.add(Input(shape=(T_, X_tr.shape[1])))
    for i in range(best['layers']):
        return_seq = i < best['layers'] - 1
        final.add(LSTM(best['units'], return_sequences=return_seq))
        final.add(Dropout(0.1))
    if classification:
        final.add(Dense(1, activation='sigmoid'))
        final.compile(loss='binary_crossentropy', optimizer='adam',
                      metrics=['accuracy'])
    else:
        final.add(Dense(1))
        final.compile(loss='mse', optimizer='adam')
    es = EarlyStopping(monitor='val_loss', patience=10,
                       restore_best_weights=True)
    final.fit(Xf3, yf3, epochs=200, batch_size=32, verbose=0,
              validation_split=0.1, callbacks=[es])
    return final, best


def predict_lstm(model, X_tr_full: np.ndarray, X_te: np.ndarray,
                 timesteps: int, classification: bool):
    """Predict on the test set, seeding the first window with the last
    (timesteps-1) rows of the train+val matrix to avoid losing observations."""
    Xte_aug = np.vstack([X_tr_full[-(timesteps - 1):], X_te])
    n_test = X_te.shape[0]
    Xs = []
    for i in range(timesteps - 1, len(Xte_aug)):
        Xs.append(Xte_aug[i - timesteps + 1:i + 1, :])
    Xs = np.array(Xs)
    Xs = Xs[-n_test:]
    raw = model.predict(Xs, verbose=0).ravel()
    if classification:
        return (raw > 0.5).astype(int)
    return raw

## Part 6: Per-target pipeline (Lasso -> scale -> NN/GP/LSTM -> evaluate)

In [ ]:
def run_target(target_kind: str, drop_var: str = None):
    """Run NN, GP, LSTM for a single target. If `drop_var` is given, all
    lags of that exogenous variable are stripped from the feature pool
    BEFORE feature selection, so the same selection-then-fit pipeline is
    applied consistently in the main run and in the ablations."""
    label = target_kind if drop_var is None else f'{target_kind}/-{drop_var}'
    print(f'\n=== Target: {label} ===')
    classification = (target_kind == 'direction')

    X, y = build_design(panel, target_kind)
    if drop_var is not None:
        X = X.drop(columns=[c for c in X.columns
                            if c.startswith(drop_var + '_lag')])
    X_tr, y_tr, X_va, y_va, X_te, y_te = chronological_split(X, y)

    keep = select_features(X_tr, y_tr, target_kind)
    print(f'  features kept ({len(keep)}): {keep}')

    X_tr = X_tr[keep].values
    X_va = X_va[keep].values
    X_te = X_te[keep].values

    X_tr_s, X_va_s, X_te_s, _ = standardise(X_tr, X_va, X_te)

    rows = []
    preds_dict = {}

    # NN
    print('  fitting NN.', flush=True)
    nn, nn_best = fit_nn(X_tr_s, y_tr, X_va_s, y_va, classification)
    p_nn = nn.predict(X_te_s)
    rows.append({'target': target_kind, 'model': 'NN', 'params': str(nn_best),
                 **evaluate(y_te, p_nn, classification)})
    preds_dict['NN'] = p_nn

    # GP
    print('  fitting GP.', flush=True)
    if classification:
        gp, gp_best = fit_gp(X_tr_s, y_tr.astype(float), X_va_s, y_va.astype(float))
        p_gp = (gp.predict(X_te_s) > 0.5).astype(int)
    else:
        gp, gp_best = fit_gp(X_tr_s, y_tr, X_va_s, y_va)
        p_gp = gp.predict(X_te_s)
    rows.append({'target': target_kind, 'model': 'GP', 'params': str(gp_best),
                 **evaluate(y_te, p_gp, classification)})
    preds_dict['GP'] = p_gp

    # LSTM
    print('  fitting LSTM.', flush=True)
    lstm, lstm_best = fit_lstm(X_tr_s, y_tr, X_va_s, y_va, classification)
    Xtrfull_s = np.vstack([X_tr_s, X_va_s])
    p_lstm = predict_lstm(lstm, Xtrfull_s, X_te_s,
                          lstm_best['timesteps'], classification)
    rows.append({'target': target_kind, 'model': 'LSTM', 'params': str(lstm_best),
                 **evaluate(y_te, p_lstm, classification)})
    preds_dict['LSTM'] = p_lstm

    # Naive baseline
    if classification:
        baseline = np.full(len(y_te), int(y_tr.mean() >= 0.5))
        rows.append({'target': target_kind, 'model': 'BaselineMajority',
                     'params': '{}', **evaluate(y_te, baseline, classification)})
        preds_dict['BaselineMajority'] = baseline
    else:
        baseline = np.full(len(y_te), float(y_tr.mean()))
        rows.append({'target': target_kind, 'model': 'BaselineMean',
                     'params': '{}', **evaluate(y_te, baseline, classification)})
        preds_dict['BaselineMean'] = baseline

    return rows, preds_dict, y_te, keep

## Part 7: Run all three target specifications (main run)

In [ ]:
print('Running the three target specifications.')
all_rows = []
all_preds = {}
all_truths = {}
all_features = {}

for target_kind in ['return', 'price', 'direction']:
    rows, preds, y_te, keep = run_target(target_kind)
    all_rows.extend(rows)
    all_preds[target_kind] = preds
    all_truths[target_kind] = y_te
    all_features[target_kind] = keep

metrics_df = pd.DataFrame(all_rows)
print('\n--- Test-set metrics (one row per target x model) ---')
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(metrics_df.to_string(index=False))

## Part 8: Sensitivity analysis (drop each exogenous variable in turn)

In [ ]:
print('Sensitivity analysis: drop each exogenous variable in turn.')
sens_rows = []
for drop_var in ASSIGNED_INDICATORS:
    for target_kind in ['return', 'price', 'direction']:
        rows, _, _, _ = run_target(target_kind, drop_var=drop_var)
        for r in rows:
            r['dropped'] = drop_var
            sens_rows.append(r)

sens_df = pd.DataFrame(sens_rows)
with pd.option_context('display.max_columns', None, 'display.width', 200):
    print(sens_df.to_string(index=False))

## Part 9: Save outputs and figures

In [ ]:
def df_to_latex(df: pd.DataFrame, name: str, caption: str):
    """pandas to_latex(index=False) emits a tabular environment;
    we wrap it once in a table environment for floating placement."""
    tex = df.to_latex(index=False, float_format='{:.4f}'.format,
                      column_format='l' * len(df.columns), escape=False)
    out = (
        '\\begin{table}[H]\n\\centering\n\\small\n'
        + tex
        + '\\caption{' + caption + '}\n\\label{tab:' + name + '}\n'
        + '\\end{table}\n'
    )
    (OUTPUT_DIR / f'{name}.tex').write_text(out)


metrics_df.to_csv(OUTPUT_DIR / 'ex2_metrics.csv', index=False)
df_to_latex(metrics_df, 'ex2_metrics',
            'Test-set forecast performance for each (target, model) combination.')

sens_df.to_csv(OUTPUT_DIR / 'ex2_sensitivity.csv', index=False)
df_to_latex(sens_df, 'ex2_sensitivity',
            'Sensitivity analysis: each row is the test-set performance when '
            'all lags of the indicated variable are removed from the feature set.')

for target_kind, preds in all_preds.items():
    y_te = all_truths[target_kind]
    out = pd.DataFrame({'y_true': y_te.values}, index=y_te.index)
    for m_name, p in preds.items():
        out[f'pred_{m_name}'] = np.asarray(p)
    out.to_csv(OUTPUT_DIR / f'ex2_pred_{target_kind}.csv')

    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.plot(y_te.index, y_te.values, label='actual', color='black', linewidth=1.4)
    style = {'NN': ('--', 'tab:orange'), 'GP': ('--', 'tab:green'),
             'LSTM': ('--', 'tab:blue'),
             'BaselineMean': (':', 'grey'),
             'BaselineMajority': (':', 'grey')}
    for m_name, p in preds.items():
        ls, c = style.get(m_name, ('--', 'tab:purple'))
        ax.plot(y_te.index, np.asarray(p), label=m_name, linestyle=ls,
                color=c, alpha=0.85, linewidth=1.2)
    ax.set_title(f'Test-set predictions: {target_kind}')
    ax.legend(loc='best', fontsize=9)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'ex2_fig_{target_kind}.pdf', bbox_inches='tight')
    plt.show()
    plt.close(fig)

fs = pd.DataFrame({k: pd.Series(v) for k, v in all_features.items()})
fs.to_csv(OUTPUT_DIR / 'ex2_feature_selection.csv', index=False)

print('Done. Artefacts written to', OUTPUT_DIR)

## Part 10: Output location

In [ ]:
# Output bundling is intentionally not performed in this notebook so that
# running the notebook in submission mode does not generate Python output
# artefacts (the assignment specifies that script output should not be
# handed in). All forecasting artefacts (CSVs, LaTeX fragments, figures)
# are written to OUTPUT_DIR by the preceding cell.
print('Run complete. Artefacts in', OUTPUT_DIR)